# Module 7 — Classical Machine Learning for Computational Materials
## Hands-On Python Tutorial

**Level:** IIT M.Tech / PhD Applied Materials / Computational Materials  
**Prerequisites:** Modules 1–6  
**Recommended duration:** 4–5 tutorials of 3 hours each

---

## Course philosophy

The objective of this module is **not** to teach machine learning as a collection of Python commands.

Students should learn to move from:

\[
\boxed{
\text{Materials problem}
\rightarrow
\text{physical descriptors}
\rightarrow
\text{dataset}
\rightarrow
\text{ML model}
\rightarrow
\text{evaluation}
\rightarrow
\text{scientific interpretation}
}
\]

The emphasis is on:

- mathematical understanding
- correct train/test methodology
- feature engineering
- model selection
- cross-validation
- hyperparameter tuning
- uncertainty and limitations
- physical interpretation
- avoiding data leakage
- reproducibility

The examples use synthetic materials datasets so the notebook can run without downloading external databases.


# Learning objectives

By the end of this module, students should be able to:

1. Explain supervised and unsupervised learning.
2. Formulate materials-property prediction as a regression problem.
3. Formulate materials classification problems.
4. Construct feature matrices \(X\) and target vectors \(y\).
5. Split data correctly into training and test sets.
6. Build baseline models.
7. Implement linear regression mathematically and with scikit-learn.
8. Understand regularization: Ridge and Lasso.
9. Train decision trees and random forests.
10. Understand nearest-neighbor regression.
11. Train support-vector regression/classification models.
12. Compare models using appropriate metrics.
13. Use cross-validation.
14. Perform hyperparameter tuning.
15. Diagnose overfitting and underfitting.
16. Interpret feature importance and model coefficients.
17. Build reproducible ML pipelines.
18. Apply classical ML to a materials-informatics problem.


# 1. Environment

We will use:

- NumPy
- Pandas
- Matplotlib
- SciPy
- scikit-learn

The notebook is designed to run in Jupyter Notebook, JupyterLab, or Google Colab.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    GridSearchCV
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    LogisticRegression
)

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    GradientBoostingRegressor
)

from sklearn.svm import SVR, SVC

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.inspection import permutation_importance

rng = np.random.default_rng(42)

print("Machine-learning environment ready.")


# 2. What is machine learning?

In supervised learning, we have examples:

\[
\{(x_i,y_i)\}_{i=1}^{N}
\]

where:

- \(x_i\) = feature vector describing a material
- \(y_i\) = measured/calculated material property

We seek a function:

\[
\hat y=f(X;\theta)
\]

that generalizes to unseen materials.

For regression:

\[
y\in\mathbb{R}.
\]

Examples:

- elastic modulus
- yield strength
- thermal conductivity
- formation energy
- band gap
- diffusion coefficient

For classification:

\[
y\in\{0,1,\ldots,K-1\}.
\]

Examples:

- stable / unstable
- metal / semiconductor / insulator
- brittle / ductile
- magnetic / non-magnetic


# 3. Materials dataset

We construct a synthetic dataset representing materials descriptors.

Features include:

- composition-related descriptor
- grain size
- density
- porosity
- processing temperature
- cooling rate
- elastic modulus
- thermal conductivity
- crystallite size
- defect concentration

The target will be tensile strength.

The dataset is intentionally designed so that students can study:

- nonlinear relationships
- correlated features
- noise
- outliers
- model bias
- feature importance


In [ ]:
n = 1000

df = pd.DataFrame({
    "mean_atomic_mass": rng.normal(70, 15, n),
    "grain_size_um": rng.lognormal(np.log(10), 0.45, n),
    "density_g_cm3": rng.normal(7.2, 0.7, n),
    "porosity_pct": np.clip(rng.normal(4, 2, n), 0, 15),
    "processing_temperature_K": rng.normal(1100, 150, n),
    "cooling_rate_K_s": rng.lognormal(np.log(20), 0.8, n),
    "elastic_modulus_GPa": rng.normal(180, 35, n),
    "thermal_conductivity_W_mK": rng.normal(80, 25, n),
    "crystallite_size_nm": rng.lognormal(np.log(80), 0.55, n),
    "defect_fraction": np.clip(rng.lognormal(np.log(0.01), 0.8, n), 0, 0.15)
})

# A physically motivated synthetic target:
# Hall-Petch-like grain-size strengthening + density/modulus effects
df["yield_strength_MPa"] = (
    180
    + 0.55 * df["elastic_modulus_GPa"]
    + 18 / np.sqrt(df["grain_size_um"])
    + 22 * df["density_g_cm3"]
    - 11 * df["porosity_pct"]
    + 0.045 * df["processing_temperature_K"]
    - 2.0 * np.log1p(df["cooling_rate_K_s"])
    - 700 * df["defect_fraction"]
    + rng.normal(0, 35, n)
)

df.head()


# 4. Exploratory data analysis

Before fitting a model, understand the dataset.

Questions:

- What are the units?
- What are the ranges?
- Are features physically plausible?
- Are there missing values?
- Are features strongly correlated?
- Is the target skewed?
- Are there suspicious outliers?


In [ ]:
df.info()


In [ ]:
df.describe().T


In [ ]:
df.isna().sum()


## Target distribution


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["yield_strength_MPa"], bins=35)
plt.xlabel("Yield strength (MPa)")
plt.ylabel("Count")
plt.title("Target distribution")
plt.grid(alpha=0.25)
plt.show()


# 5. Feature matrix and target vector

For supervised learning:

\[
X=
\begin{bmatrix}
x_{11}&x_{12}&\cdots&x_{1p}\\
x_{21}&x_{22}&\cdots&x_{2p}\\
\vdots&\vdots&&\vdots\\
x_{N1}&x_{N2}&\cdots&x_{Np}
\end{bmatrix}
\]

and:

\[
y=
\begin{bmatrix}
y_1\\
y_2\\
\vdots\\
y_N
\end{bmatrix}.
\]


In [ ]:
target = "yield_strength_MPa"

X = df.drop(columns=[target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


# 6. Train-test split

A test set should represent **unseen materials**.

We use:

\[
\text{training set}\rightarrow\text{fit model}
\]

\[
\text{test set}\rightarrow\text{final evaluation}.
\]

The test set should not be used repeatedly for model selection.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


# 7. Baseline model

Before sophisticated models, establish a baseline.

A simple baseline predicts the training-set mean:

\[
\hat y_i=\bar y_{\mathrm{train}}.
\]

Any useful ML model should outperform this baseline on unseen data.


In [ ]:
baseline_prediction = np.full(
    len(y_test),
    y_train.mean()
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_prediction
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_prediction
)

print("Baseline MAE :", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²  :", baseline_r2)


# 8. Evaluation metrics

For regression:

### Mean absolute error

\[
MAE=
\frac1N
\sum_i|y_i-\hat y_i|.
\]

### Root mean squared error

\[
RMSE=
\sqrt{
\frac1N
\sum_i(y_i-\hat y_i)^2
}.
\]

### Coefficient of determination

\[
R^2=
1-
\frac{\sum_i(y_i-\hat y_i)^2}
{\sum_i(y_i-\bar y)^2}.
\]

RMSE penalizes large errors more strongly than MAE.

Always report metrics in the physical units of the target where possible.


# 9. Linear regression

The linear model is:

\[
\hat y=
\beta_0+
\beta_1x_1+\cdots+\beta_px_p.
\]

In matrix form:

\[
\hat{\mathbf y}=X\boldsymbol\beta.
\]

Ordinary least squares minimizes:

\[
J(\beta)=
\|X\beta-y\|_2^2.
\]

The normal-equation solution is:

\[
\hat\beta=
(X^TX)^{-1}X^Ty
\]

when the inverse exists and the formulation is numerically appropriate.

In practice, stable linear-algebra algorithms are preferred over explicitly computing the inverse.


In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_linear))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_linear)))
print("R²  :", r2_score(y_test, y_pred_linear))


# 10. Linear regression from NumPy

For educational purposes, construct the design matrix and solve the least-squares problem using `np.linalg.lstsq`.


In [ ]:
X_train_np = X_train.to_numpy()
X_test_np = X_test.to_numpy()

X_train_design = np.column_stack([
    np.ones(len(X_train_np)),
    X_train_np
])

X_test_design = np.column_stack([
    np.ones(len(X_test_np)),
    X_test_np
])

beta, residuals, rank, singular_values = np.linalg.lstsq(
    X_train_design,
    y_train.to_numpy(),
    rcond=None
)

y_pred_manual = X_test_design @ beta

print("Manual least-squares coefficients:")
print(beta)

print("
Manual model performance:")
print("MAE :", mean_absolute_error(y_test, y_pred_manual))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_manual)))
print("R²  :", r2_score(y_test, y_pred_manual))


# Exercise 1 — Linear regression

1. Compare the coefficients from `LinearRegression` and `np.linalg.lstsq`.
2. Identify which features have the largest absolute coefficients.
3. Explain why coefficient magnitude cannot always be compared directly when features have different units.
4. Standardize the features and repeat the regression.
5. Compare standardized coefficients.


# 11. Residual analysis

A model is not fully characterized by one number.

Define residuals:

\[
e_i=y_i-\hat y_i.
\]

Inspect:

- residual distribution
- residual versus prediction
- residual versus feature
- systematic trends


In [ ]:
residuals = y_test.to_numpy() - y_pred_linear

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_linear, residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted strength (MPa)")
plt.ylabel("Residual (MPa)")
plt.title("Linear-regression residuals")
plt.grid(alpha=0.25)
plt.show()


# 12. Ridge regression

Ridge adds an \(L_2\) penalty:

\[
J(\beta)
=
\|X\beta-y\|_2^2
+
\lambda\|\beta\|_2^2.
\]

The penalty discourages large coefficients.

Ridge is useful when:

- features are correlated
- the problem is ill-conditioned
- some degree of shrinkage improves generalization


In [ ]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10.0))
])

ridge_pipeline.fit(X_train, y_train)

y_pred_ridge = ridge_pipeline.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_ridge))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ridge)))
print("R²  :", r2_score(y_test, y_pred_ridge))


# 13. Lasso regression

Lasso uses an \(L_1\) penalty:

\[
J(\beta)
=
\|X\beta-y\|_2^2
+
\lambda\|\beta\|_1.
\]

The \(L_1\) penalty can drive some coefficients exactly to zero.

Therefore Lasso can perform a form of feature selection.

However, feature selection should be treated carefully when descriptors are highly correlated.


In [ ]:
lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.05, max_iter=20000))
])

lasso_pipeline.fit(X_train, y_train)

y_pred_lasso = lasso_pipeline.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_lasso))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lasso)))
print("R²  :", r2_score(y_test, y_pred_lasso))


# 14. Compare linear models


In [ ]:
models = {
    "Baseline": baseline_prediction,
    "Linear": y_pred_linear,
    "Ridge": y_pred_ridge,
    "Lasso": y_pred_lasso
}

comparison = []

for name, pred in models.items():
    comparison.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

pd.DataFrame(comparison).sort_values("RMSE")


# 15. k-Nearest Neighbors regression

For a new material, KNN finds nearby training materials in feature space.

Prediction:

\[
\hat y(x)
=
\frac{1}{k}
\sum_{i\in N_k(x)}y_i.
\]

Because distances are used, feature scaling is essential.


In [ ]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsRegressor(n_neighbors=10))
])

knn_pipeline.fit(X_train, y_train)

y_pred_knn = knn_pipeline.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_knn))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_knn)))
print("R²  :", r2_score(y_test, y_pred_knn))


# Exercise 2 — Effect of \(k\)

Train KNN models with:

\[
k=1,3,5,10,20,40,80.
\]

Plot training and test errors against \(k\).

Interpret the result in terms of:

- high variance
- high bias
- overfitting
- underfitting


In [ ]:
k_values = [1, 3, 5, 10, 20, 40, 80]

train_rmse = []
test_rmse = []

for k in k_values:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor(n_neighbors=k))
    ])

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_rmse.append(
        np.sqrt(mean_squared_error(y_train, train_pred))
    )
    test_rmse.append(
        np.sqrt(mean_squared_error(y_test, test_pred))
    )

plt.figure(figsize=(8, 5))
plt.plot(k_values, train_rmse, "o-", label="Train RMSE")
plt.plot(k_values, test_rmse, "s-", label="Test RMSE")
plt.xlabel("k")
plt.ylabel("RMSE")
plt.title("KNN bias–variance behavior")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 16. Decision trees

A decision tree recursively partitions feature space.

For regression, a leaf predicts a representative target value, typically the mean of training targets reaching that leaf.

Trees can represent nonlinear relationships without explicitly specifying a functional form.


In [ ]:
tree = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_tree))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_tree)))
print("R²  :", r2_score(y_test, y_pred_tree))


# 17. Random forests

A random forest averages predictions from many decision trees.

Conceptually:

\[
\hat y(x)
=
\frac1B
\sum_{b=1}^{B}
\hat y_b(x).
\]

Randomization and averaging generally reduce the variance of an individual tree.


In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R²  :", r2_score(y_test, y_pred_rf))


# 18. Gradient boosting

Gradient boosting constructs models sequentially.

At each stage, the new model attempts to improve the errors made by the previous ensemble.

This can provide excellent performance on structured/tabular materials data.


In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    random_state=42
)

gbr.fit(X_train, y_train)

y_pred_gbr = gbr.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_gbr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_gbr)))
print("R²  :", r2_score(y_test, y_pred_gbr))


# 19. Support-vector regression

SVR attempts to fit a function that is insensitive to errors smaller than \(\epsilon\), while controlling model complexity.

The kernel trick allows nonlinear relationships to be represented.

For RBF-SVR, important hyperparameters include:

- \(C\)
- \(\gamma\)
- \(\epsilon\)

Feature scaling is essential.


In [ ]:
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(
        kernel="rbf",
        C=100,
        gamma="scale",
        epsilon=0.1
    ))
])

svr_pipeline.fit(X_train, y_train)

y_pred_svr = svr_pipeline.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_svr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_svr)))
print("R²  :", r2_score(y_test, y_pred_svr))


# 20. Model comparison

Now compare several classical models on the same held-out test set.


In [ ]:
prediction_dict = {
    "Linear": y_pred_linear,
    "Ridge": y_pred_ridge,
    "Lasso": y_pred_lasso,
    "KNN": y_pred_knn,
    "Decision Tree": y_pred_tree,
    "Random Forest": y_pred_rf,
    "Gradient Boosting": y_pred_gbr,
    "SVR": y_pred_svr
}

rows = []

for name, pred in prediction_dict.items():
    rows.append({
        "Model": name,
        "MAE_MPa": mean_absolute_error(y_test, pred),
        "RMSE_MPa": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

results = pd.DataFrame(rows).sort_values("RMSE_MPa")
results


# 21. Predicted versus experimental values

A useful diagnostic is:

\[
y_{\mathrm{pred}}\quad\text{vs}\quad y_{\mathrm{true}}.
\]

A perfect model lies on:

\[
y_{\mathrm{pred}}=y_{\mathrm{true}}.
\]


In [ ]:
best_model_name = results.iloc[0]["Model"]
best_pred = prediction_dict[best_model_name]

plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_pred, alpha=0.6)

limits = [
    min(y_test.min(), best_pred.min()),
    max(y_test.max(), best_pred.max())
]

plt.plot(limits, limits, "--")

plt.xlabel("True yield strength (MPa)")
plt.ylabel("Predicted yield strength (MPa)")
plt.title(f"Best model: {best_model_name}")
plt.grid(alpha=0.25)
plt.show()


# 22. Cross-validation

A single train/test split can produce a lucky or unlucky result.

In \(K\)-fold cross-validation:

1. split training data into \(K\) folds
2. train on \(K-1\)
3. validate on the remaining fold
4. repeat
5. aggregate performance

The test set remains untouched until final evaluation.


In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_cv = cross_validate(
    rf,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    return_train_score=True,
    n_jobs=-1
)

cv_summary = pd.DataFrame({
    "Train_RMSE": -rf_cv["train_RMSE"],
    "Validation_RMSE": -rf_cv["test_RMSE"],
    "Validation_R2": rf_cv["test_R2"]
})

cv_summary


# 23. Cross-validation interpretation

Look for:

- mean performance
- spread between folds
- train/validation gap

A large train-validation gap often indicates overfitting.

However, the split strategy itself matters enormously for materials data.

If multiple rows correspond to the same:

- composition
- material
- processing batch
- simulation structure
- chemical family

then random row-wise splitting may leak information across folds.

This is one of the most important issues in materials machine learning.


# 24. Data leakage demonstration

Imagine that each composition has five measurements.

If the same composition appears in both training and test sets, the model may appear highly accurate because it has effectively seen related samples.

A scientifically meaningful split may instead require:

- group splitting by composition
- splitting by material family
- time-based splitting
- leave-one-composition-out validation

The correct split depends on the intended deployment scenario.


# 25. Hyperparameter tuning

Hyperparameters are not learned directly from the training objective.

Examples:

- number of trees
- tree depth
- \(k\) in KNN
- regularization strength
- SVR \(C\)
- kernel parameters

We can use cross-validation to select them.


In [ ]:
param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [None, 8, 15],
    "min_samples_leaf": [1, 2, 5]
}

grid_search = GridSearchCV(
    RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("
Best CV RMSE:")
print(-grid_search.best_score_)


In [ ]:
tuned_rf = grid_search.best_estimator_

tuned_pred = tuned_rf.predict(X_test)

print("Test MAE :", mean_absolute_error(y_test, tuned_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, tuned_pred)))
print("Test R²  :", r2_score(y_test, tuned_pred))


# 26. Feature importance

Tree models provide impurity-based feature importance.

These values can be useful, but they should **not automatically be interpreted as causal physical importance**.

Correlated features can share or distort importance.


In [ ]:
importance = pd.Series(
    tuned_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
importance.sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.title("Random-forest feature importance")
plt.grid(axis="x", alpha=0.25)
plt.show()

importance


# 27. Permutation importance

Permutation importance asks:

> How much does model performance degrade if this feature is randomly shuffled?

This is often more directly connected to predictive usefulness than impurity importance.


In [ ]:
perm = permutation_importance(
    tuned_rf,
    X_test,
    y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_importance = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
perm_importance.sort_values().plot(kind="barh")
plt.xlabel("Mean decrease in score")
plt.title("Permutation importance")
plt.grid(axis="x", alpha=0.25)
plt.show()


# 28. Linear-model interpretation

For a standardized linear model:

\[
\hat y=
\beta_0+
\sum_j\beta_jz_j,
\]

where \(z_j\) are standardized features.

The magnitude of \(\beta_j\) indicates how strongly the prediction changes per one standard deviation change in the feature, **holding other features fixed**.

This is useful for scientific interpretation, but correlation between descriptors must be considered.


In [ ]:
standardized_linear = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

standardized_linear.fit(X_train, y_train)

coef = pd.Series(
    standardized_linear.named_steps["model"].coef_,
    index=X.columns
).sort_values()

plt.figure(figsize=(9, 5))
coef.plot(kind="barh")
plt.xlabel("Standardized coefficient")
plt.title("Linear-model standardized coefficients")
plt.grid(axis="x", alpha=0.25)
plt.show()


# 29. Classification problem

We now convert the problem into a classification task.

Suppose we want to identify whether a material has:

\[
\sigma_y > 600\ \mathrm{MPa}.
\]

Define:

\[
y_{\mathrm{class}}=
\begin{cases}
1,&\sigma_y>600\\
0,&\text{otherwise}.
\end{cases}
\]


In [ ]:
df["high_strength"] = (
    df["yield_strength_MPa"] > 600
).astype(int)

X_cls = df.drop(
    columns=["yield_strength_MPa", "high_strength"]
)

y_cls = df["high_strength"]

print(y_cls.value_counts())


# 30. Classification split


In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls,
    y_cls,
    test_size=0.20,
    stratify=y_cls,
    random_state=42
)


# 31. Logistic regression

Logistic regression models:

\[
P(y=1|x)
=
\frac{1}
{1+\exp[-(\beta_0+\beta^Tx)]}.
\]

The model produces a probability rather than simply a class label.


In [ ]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

logistic_pipeline.fit(Xc_train, yc_train)

yc_pred = logistic_pipeline.predict(Xc_test)
yc_prob = logistic_pipeline.predict_proba(Xc_test)[:, 1]

print("Accuracy :", accuracy_score(yc_test, yc_pred))
print("Precision:", precision_score(yc_test, yc_pred))
print("Recall   :", recall_score(yc_test, yc_pred))
print("F1       :", f1_score(yc_test, yc_pred))


# 32. Confusion matrix

For binary classification:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | TN | FP |
| Actual 1 | FN | TP |

The relative importance of false positives and false negatives depends on the materials application.


In [ ]:
cm = confusion_matrix(yc_test, yc_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)

disp.plot()
plt.title("Logistic-regression confusion matrix")
plt.show()


# 33. Random-forest classification


In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(Xc_train, yc_train)

yc_rf_pred = rf_classifier.predict(Xc_test)

print("Accuracy :", accuracy_score(yc_test, yc_rf_pred))
print("Precision:", precision_score(yc_test, yc_rf_pred))
print("Recall   :", recall_score(yc_test, yc_rf_pred))
print("F1       :", f1_score(yc_test, yc_rf_pred))


# Exercise 3 — Classification

Train and compare:

- Logistic regression
- Decision tree
- Random forest
- SVM classifier

Report:

- accuracy
- precision
- recall
- F1 score
- confusion matrix

Then answer:

> Which metric should matter most if missing a genuinely high-strength material is more costly than testing an ordinary material unnecessarily?


# 34. Pipelines and preprocessing

A major source of leakage occurs when preprocessing is performed before splitting the data.

For example, this is dangerous:

```python
scaler.fit_transform(X)
train_test_split(...)
```

The scaler has seen information from the eventual test set.

Instead:

```text
split
  ↓
fit preprocessing on training data
  ↓
transform training data
transform test data
  ↓
fit model
```

A scikit-learn `Pipeline` automates this safely.


# 35. Add a categorical materials descriptor

Real materials datasets contain categorical information:

- crystal system
- processing route
- alloy family
- phase label
- synthesis method

We add a synthetic categorical variable.


In [ ]:
categories = np.array([
    "FCC",
    "BCC",
    "HCP"
])

df["crystal_structure"] = rng.choice(
    categories,
    size=len(df),
    p=[0.45, 0.35, 0.20]
)

df["processing_route"] = rng.choice(
    ["cast", "annealed", "forged"],
    size=len(df)
)

df[[
    "crystal_structure",
    "processing_route",
    "yield_strength_MPa"
]].head()


# 36. Mixed numerical + categorical preprocessing

A robust materials ML workflow may require:

\[
\text{numerical features}
\rightarrow
\text{imputation/scaling}
\]

and:

\[
\text{categorical features}
\rightarrow
\text{imputation/one-hot encoding}.
\]

`ColumnTransformer` lets us combine them.


In [ ]:
target = "yield_strength_MPa"

X_mixed = df.drop(columns=[target])
y_mixed = df[target]

numeric_features = X_mixed.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_mixed.select_dtypes(
    exclude=np.number
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

mixed_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=250,
        random_state=42,
        n_jobs=-1
    ))
])

Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_mixed,
    y_mixed,
    test_size=0.20,
    random_state=42
)

mixed_model.fit(Xm_train, ym_train)

ym_pred = mixed_model.predict(Xm_test)

print("MAE :", mean_absolute_error(ym_test, ym_pred))
print("RMSE:", np.sqrt(mean_squared_error(ym_test, ym_pred)))
print("R²  :", r2_score(ym_test, ym_pred))


# 37. A realistic materials-ML workflow

A strong workflow is:

```text
Materials question
      ↓
Define target property
      ↓
Collect data
      ↓
Check provenance
      ↓
Clean data
      ↓
Define physically meaningful descriptors
      ↓
Split data appropriately
      ↓
Build baseline
      ↓
Cross-validation
      ↓
Train candidate models
      ↓
Tune hyperparameters
      ↓
Evaluate on untouched test set
      ↓
Interpret model
      ↓
Check physical plausibility
      ↓
Quantify uncertainty / limitations
      ↓
Use model for prediction or screening
```

This workflow connects Modules 5, 6, and 7.


# 38. Materials-specific data splitting

Random splitting is not always scientifically valid.

Consider a dataset containing:

```text
Al-1
Al-2
Al-3
...
Al-100
```

where all samples belong to nearly the same alloy family.

A random split may answer:

> Can the model predict another sample from a familiar family?

But your scientific question might be:

> Can the model predict an entirely new alloy family?

These are different generalization problems.

Possible strategies include:

- random split
- group split
- leave-one-group-out
- composition-based split
- temporal split
- extrapolation-oriented split


# 39. Feature engineering for materials

Good ML begins before model fitting.

Examples of physically meaningful descriptors:

### Grain strengthening

\[
\sigma_y=
\sigma_0+k_y d^{-1/2}.
\]

A useful descriptor is:

\[
d^{-1/2}.
\]

### Thermal activation

\[
\exp(-Q/RT).
\]

### Diffusion length

\[
\ell\sim\sqrt{Dt}.
\]

### Composition descriptors

Examples:

- mean atomic radius
- atomic-size mismatch
- electronegativity statistics
- valence-electron concentration
- elemental fractions

Later, `pymatgen` and `matminer` can automate many materials descriptors.


In [ ]:
df["hall_petch_descriptor"] = (
    1 / np.sqrt(df["grain_size_um"])
)

df["specific_modulus"] = (
    df["elastic_modulus_GPa"]
    / df["density_g_cm3"]
)

df[[
    "grain_size_um",
    "hall_petch_descriptor",
    "specific_modulus"
]].head()


# 40. Model complexity and the bias–variance trade-off

A model can be:

### Underfit

Too simple to capture the underlying relationship.

High bias.

### Overfit

Fits noise and idiosyncrasies of the training data.

High variance.

The goal is not:

\[
\text{maximum training accuracy}.
\]

The goal is:

\[
\boxed{\text{good generalization to the intended unseen materials}}
\]


# 41. Learning curves — suggested exercise

Use `sklearn.model_selection.learning_curve` to compare:

- linear regression
- random forest
- gradient boosting

Plot training and validation error against the number of training samples.

Interpret whether collecting more data is likely to improve performance.


# 42. Important scientific caution

A high \(R^2\) does not prove that:

- the model discovered a physical law
- the descriptors are causal
- the model extrapolates
- the data are unbiased
- the predictions are reliable outside the training domain

A model can interpolate extremely well while failing catastrophically in extrapolation.

For computational materials, **domain of applicability** is as important as the average prediction error.


# 43. Mini-project — Materials property prediction

## Objective

Build a complete ML workflow to predict a materials property.

Possible targets:

- yield strength
- elastic modulus
- thermal conductivity
- hardness
- formation energy
- band gap

### Required steps

1. Define the scientific question.
2. Describe the dataset.
3. Perform exploratory analysis.
4. Define features and target.
5. Choose a scientifically justified train/test strategy.
6. Establish a baseline.
7. Train at least four models.
8. Use cross-validation.
9. Tune at least one model.
10. Compare metrics.
11. Analyze residuals.
12. Analyze feature importance.
13. Discuss physical interpretation.
14. Identify the domain of applicability.
15. Identify likely failure modes.

### Required models

At minimum:

- Linear/Ridge regression
- KNN
- Random forest
- Gradient boosting or SVR


# 44. Mini-project — Materials classification

Predict whether a material belongs to a target class.

Examples:

- high-strength / low-strength
- metal / non-metal
- stable / unstable
- magnetic / non-magnetic
- ductile / brittle

Requirements:

- justify the classification threshold
- handle class imbalance if present
- report precision, recall and F1
- show confusion matrix
- compare at least three classifiers
- explain the consequences of false positives and false negatives


# 45. Capstone connection

This module prepares students for a full materials-informatics capstone:

\[
\boxed{
\text{Materials database}
\rightarrow
\text{descriptors}
\rightarrow
\text{data cleaning}
\rightarrow
\text{feature engineering}
\rightarrow
\text{ML}
\rightarrow
\text{candidate screening}
}
\]

A particularly strong capstone can connect numerical simulation with ML:

\[
\text{PDE simulation}
\rightarrow
\text{simulation dataset}
\rightarrow
\text{features}
\rightarrow
\text{ML surrogate}
\]

For example:

\[
\text{heat/diffusion simulation}
\rightarrow
T(x,t),C(x,t)
\rightarrow
\text{summary descriptors}
\rightarrow
\text{predict processing outcome}.
\]


# 46. Recommended libraries for materials machine learning

### Core ML

- `scikit-learn`

### Materials structures and chemistry

- `pymatgen`
- `matminer`
- `ASE`

### Scientific computing

- `NumPy`
- `SciPy`
- `Pandas`

### Visualization

- `Matplotlib`
- `Seaborn`

### Larger-scale ML / deep learning — later modules

- `XGBoost` / `LightGBM` where appropriate
- `PyTorch`
- `TensorFlow` / Keras

### Data and experiment tracking

- `joblib`
- MLflow or equivalent experiment tracking tools

Students should first master the underlying concepts before introducing large ML frameworks.


# 47. Assessment

## Conceptual

Students should be able to explain:

- supervised vs unsupervised learning
- bias vs variance
- train/validation/test sets
- cross-validation
- regularization
- hyperparameters
- data leakage
- interpolation vs extrapolation

## Computational

Students should be able to:

- construct a reproducible ML pipeline
- preprocess data
- train several models
- perform cross-validation
- tune hyperparameters
- calculate appropriate metrics
- visualize predictions and residuals

## Scientific

Students should be able to:

- justify descriptors physically
- choose a meaningful data split
- interpret model limitations
- distinguish correlation from causation
- assess whether predictions are physically plausible


# 48. Final exercise — build your own model

Choose one materials property and answer:

> **Can classical machine learning predict this property accurately enough to be useful for materials screening?**

Your notebook should contain:

1. Scientific motivation
2. Dataset description
3. Data-quality analysis
4. Descriptor selection
5. Train/test methodology
6. Baseline
7. At least four ML models
8. Cross-validation
9. Hyperparameter tuning
10. Final test evaluation
11. Residual analysis
12. Feature interpretation
13. Error analysis
14. Physical interpretation
15. Limitations
16. Reproducibility statement

### Final question

Do not finish with:

> "Random forest gave \(R^2=0.92\)."

Finish with:

> "Under these data-splitting assumptions and within this domain of materials, the model appears capable/incapable of supporting ___ because ___."

That is the level of reasoning expected in computational materials research.


# 49. Key takeaways

\[
\boxed{
\text{Good ML starts with a good scientific question}
}
\]

\[
\boxed{
\text{Descriptors encode scientific assumptions}
}
\]

\[
\boxed{
\text{Data splitting defines what "generalization" means}
}
\]

\[
\boxed{
\text{Cross-validation is for model development, not a replacement for a final test set}
}
\]

\[
\boxed{
\text{A strong score does not guarantee physical validity}
}
\]

\[
\boxed{
\text{Interpretability and domain of applicability matter in materials science}
}
\]

The next stage is to move from classical ML toward **advanced machine learning, deep learning, surrogate modelling, and materials informatics**.
